# Resumo do TCC MBA — Avaliação de IA Generativa em Saúde Pública (Malária)

Este notebook é um **resumo para portfólio** do meu trabalho final de MBA (USP/Esalq, 2025).  
Ele apresenta metodologia, principais resultados e implicações práticas usando as saídas processadas geradas pelo script original:

- `code/TCC_MBA_DSA_USP_MALARIA.py`

> Observação: o relatório completo foi escrito em português; este notebook mantém o foco executivo para revisão rápida no portfólio.


## 1. Contexto do projeto

**Pergunta de pesquisa**

Quão próximas estão as respostas dos sistemas de IA generativa em relação ao conteúdo oficial da OMS sobre malária, considerando legibilidade, similaridade lexical/semântica e consistência entre tópicos?

**Desenho dos dados**

- 10 sistemas de IA comparados com respostas de referência da OMS
- 12 perguntas distribuídas em 4 grupos temáticos
- Unidade analítica final para modelagem por tópico: `IA × tópico`


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

BASE = Path("..")
RESULTS = BASE / "results"
FIGURES = BASE / "figures"

df_agg_ranked = pd.read_excel(RESULTS / "df_agg_ranked.xlsx")
df_agg_topic = pd.read_excel(RESULTS / "df_agg_topic.xlsx")
df_agg_topic_cluster = pd.read_excel(RESULTS / "df_agg_topic_cluster.xlsx")
df_oms_metrics = pd.read_excel(RESULTS / "df_oms_metrics.xlsx")
df_all_responses = pd.read_excel(RESULTS / "df_all_responses.xlsx")

print("Loaded:")
print("- df_agg_ranked:", df_agg_ranked.shape)
print("- df_agg_topic:", df_agg_topic.shape)
print("- df_agg_topic_cluster:", df_agg_topic_cluster.shape)
print("- df_oms_metrics:", df_oms_metrics.shape)
print("- df_all_responses:", df_all_responses.shape)

In [ ]:
print("AI models in evaluation:", sorted(df_agg_ranked["AI"].unique()))
print("Topics:", sorted(df_agg_topic["topic"].unique()))
print("Questions per topic (OMS):")
display(df_oms_metrics.groupby("topic")["question"].count().rename("n_questions").reset_index())

## 2. Metodologia (visão executiva)

O estudo original combina análise de texto e estatística multivariada:

1. **Legibilidade:** Flesch Reading Ease, Flesch-Kincaid Grade Level  
2. **Similaridade:** Cosseno (TF-IDF), Jaccard, distância de Levenshtein  
3. **Ranking composto:** normalização min-max + score agregado  
4. **Análise por tópico:** ranking por tópico (`IA × tópico`)  
5. **Aprendizado não supervisionado:** K-Means (`k` definido por Elbow + Silhouette)  
6. **Validação e interpretação:** ANOVA, qui-quadrado e MCA (mapas perceptuais 2D/3D)


## 3. Ranking geral das IAs

A tabela abaixo mostra o score final normalizado (`score_final`) usado para ordenar os sistemas no cenário global (todas as métricas combinadas).


In [ ]:
ranking_cols = [
    "ranking",
    "AI",
    "score_final",
    "flesch_reading_ease",
    "cosine_similarity",
    "jaccard_similarity",
]

df_rank = df_agg_ranked.sort_values("ranking").copy()
display(df_rank[ranking_cols])

top3 = df_rank.nsmallest(3, "ranking")[["ranking", "AI", "score_final"]]
print("Top 3 systems by composite score:")
for _, r in top3.iterrows():
    print(f"#{int(r['ranking'])} {r['AI']} ? score_final={r['score_final']:.3f}")

### Interpretação para portfólio

- O ranking é **relativo a este desenho de benchmark**, não um leaderboard universal.
- Os melhores modelos foram os que equilibraram **legibilidade + similaridade semântica + alinhamento lexical** com o conteúdo da OMS.


## 4. Desempenho por tópico

O ranking global esconde especializações temáticas. A tabela abaixo mostra o melhor sistema por tópico usando `score_topic`.


In [ ]:
best_by_topic = (
    df_agg_topic.sort_values(["topic", "score_topic"], ascending=[True, False])
    .groupby("topic", as_index=False)
    .first()[["topic", "AI", "score_topic"]]
    .rename(columns={"AI": "best_AI"})
)
display(best_by_topic)

topic_spread = (
    df_agg_topic.groupby("topic")["score_topic"]
    .agg(["min", "max", "mean", "std"])
    .reset_index()
)
display(topic_spread)

## 5. Clusterização e evidência estatística

No projeto original, `k=3` clusters foi selecionado com base em Elbow + Silhouette, com evidência de diferenças entre grupos.

Abaixo, reproduzimos uma verificação estatística compacta usando a tabela exportada de tópico-cluster.


In [ ]:
cluster_summary = (
    df_agg_topic_cluster.groupby("Cluster")["score_topic"]
    .agg(["count", "mean", "median", "std"])
    .reset_index()
    .sort_values("mean")
)
display(cluster_summary)

# One-way ANOVA on topic scores across clusters
groups = [
    df_agg_topic_cluster.loc[df_agg_topic_cluster["Cluster"] == c, "score_topic"].values
    for c in sorted(df_agg_topic_cluster["Cluster"].unique())
]
F_stat, p_value = f_oneway(*groups)
print(f"ANOVA on score_topic by Cluster: F={F_stat:.4f}, p-value={p_value:.6g}")

## 6. Destaques visuais

Gráficos representativos gerados pelo script original:

- Ranking final
- Ranking por tópico
- Métodos Elbow e Silhouette
- Mapa perceptual MCA em 2D


In [ ]:
from matplotlib.image import imread

img_files = [
    "08_ai_final_ranking.png",
    "10_ai_ranking_by_topic.png",
    "11_elbow_method.png",
    "12_silhouette_method.png",
    "15_mca_perceptual_map_2d.png",
]

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
axes = axes.flatten()

for ax, name in zip(axes, img_files):
    p = FIGURES / name
    if p.exists():
        ax.imshow(imread(p))
        ax.set_title(name)
        ax.axis("off")
    else:
        ax.text(0.5, 0.5, f"Missing: {name}", ha="center", va="center")
        ax.axis("off")

# Hide any remaining empty subplot
for j in range(len(img_files), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

## 7. Conclusões finais

- A metodologia foi eficaz para distinguir comportamento **estável vs. instável** de modelos em um contexto sensível de informação em saúde.
- Os resultados indicam que a **formulação temática das perguntas** influencia fortemente os padrões de resposta.
- A combinação de métricas complementares + clusterização + testes inferenciais forma um framework robusto para auditoria de respostas de IA generativa.

## 8. Reprodutibilidade e artefatos completos

- Implementação completa: `../code/TCC_MBA_DSA_USP_MALARIA.py`
- Dados brutos/processados: `../data/` e `../results/`
- Figuras: `../figures/`

Este notebook é intencionalmente conciso e foi pensado para comunicação em portfólio.
